# Cumulative Net Invested Capital Over Time

This notebook calculates cumulative net invested capital from the normalized I/O definition. Inflows add their operation value and outflows subtract it. Missing operation values are ignored.

This is a cash-flow measure, not the cost basis of the remaining positions.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.core.models.transaction import IOFlow
from src.core.repositories.transaction_repository import TransactionRepository
from src.core.services.portfolio_service import PortfolioService

In [ ]:
transactions = TransactionRepository().all()
portfolio_history = PortfolioService().get_over_time()
portfolio_value_table = pd.DataFrame([
    {
        'date': portfolio_value.evaluation_date,
        'portfolio_value': portfolio_value.value,
    }
    for portfolio_value in portfolio_history
])

cash_flow_table = (
    pd.DataFrame([
        {
            'date': transaction.date,
            'signed_io_value': (
                transaction.operation_value
                if transaction.io_flow is IOFlow.INFLOW
                else -transaction.operation_value
            ),
        }
        for transaction in transactions
        if transaction.operation_value is not None
    ], columns=['date', 'signed_io_value'])
    .groupby('date', as_index=False)['signed_io_value']
    .sum()
)

if cash_flow_table.empty:
    cash_flow_table['invested_capital'] = pd.Series(dtype='object')
else:
    cash_flow_table = cash_flow_table.sort_values('date')
    cash_flow_table['invested_capital'] = cash_flow_table['signed_io_value'].cumsum()

chart_table = (
    portfolio_value_table[['date', 'portfolio_value']]
    .merge(
        cash_flow_table[['date', 'invested_capital']],
        on='date',
        how='outer',
    )
    .sort_values('date')
    .assign(invested_capital=lambda table: table['invested_capital'].ffill().fillna(0))
)

display(chart_table.tail())

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(
    chart_table['date'],
    chart_table['invested_capital'].astype(float),
    linewidth=2,
    color='#2a9d8f',
    label='Cumulative net invested capital',
)
plt.plot(
    chart_table['date'],
    chart_table['portfolio_value'].astype(float),
    linewidth=2,
    color='#264653',
    label='Portfolio value',
)
plt.title('Portfolio Value and Cumulative Net Invested Capital')
plt.xlabel('Date')
plt.ylabel('Amount')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()